# Dataset NTP 900.058 – 7 colores con splits train/val/test
Descarga **Garbage Dataset V2** (Kaggle) y **TrashBox** (GitHub), reparte cada imagen en su color,
balancea, separa en **70 / 15 / 15** y lo guarda en tu Drive en `data basurita/splits/`.

Tus fotos propias que estén en `data basurita/Azul`, `data basurita/Rojo`, etc. también se reparten en los splits.

No necesita GPU: `Entorno de ejecución → CPU` está bien. Tarda ~15–25 min (lo más lento es copiar a Drive).

## 1. Instalar y conectar Drive

In [ ]:
!pip install -q kagglehub
from google.colab import drive
drive.mount('/content/drive')

## 2. Configuración
- `DRIVE_DIR`: tu carpeta de Drive (la de la captura).
- `MAX_POR_COLOR`: tope de imágenes descargadas por color, para que ninguno aplaste a los demás.
- `LADO_MAX`: las imágenes se reducen a este lado máximo (px) para que la copia a Drive sea rápida.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/data basurita"
SALIDA_LOCAL = "/content/splits"
MAX_POR_COLOR = 2000
PROPORCION = {"train": 0.70, "val": 0.15, "test": 0.15}
LADO_MAX = 512
SEMILLA = 42

COLORES = ["Azul", "Amarillo", "Blanco", "Gris", "Marron", "Rojo", "Negro"]
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

## 3. Descargar los datasets

In [ ]:
import kagglehub, subprocess, os

# Garbage Dataset V2 (Kaggle) - fotos reales con fondos variados
RUTA_GD = kagglehub.dataset_download("sumn2u/garbage-classification-v2")
print("Garbage V2 en:", RUTA_GD)

# TrashBox (GitHub) - solo la carpeta con subclases, sin el resto del repo
RUTA_TB = "/content/TrashBox"
if not os.path.isdir(RUTA_TB):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/nikhilvenkatkumsetty/TrashBox", RUTA_TB], check=True)
    subprocess.run(["git", "-C", RUTA_TB, "sparse-checkout", "set",
                    "TrashBox_train_dataset_subfolders"], check=True)
RUTA_TB = f"{RUTA_TB}/TrashBox_train_dataset_subfolders"
print("TrashBox en:", RUTA_TB)

## 4. Mapeo a los 7 colores
Cada regla dice: *esta carpeta del dataset → este color*. TrashBox se mapea por subcarpeta,
así los **vasos de papel y colillas van a Negro** y los **aerosoles a Rojo**. Cámbialo si tu profe usa otro criterio.

In [ ]:
# Garbage Dataset V2: nombre de carpeta -> color
MAPEO_GD = {
    "paper": "Azul", "cardboard": "Azul",
    "metal": "Amarillo",
    "plastic": "Blanco",
    "glass": "Gris",
    "biological": "Marron",
    "battery": "Rojo",
    "trash": "Negro", "shoes": "Negro", "clothes": "Negro",
}

# TrashBox: ruta relativa (carpeta/subcarpeta) -> color
MAPEO_TB = {
    "cardboard": "Azul",
    "paper/paper": "Azul", "paper/news paper": "Azul", "paper/tetra pak": "Azul",
    "paper/paper cups": "Negro",
    "metal/beverage cans": "Amarillo", "metal/metal containers": "Amarillo",
    "metal/construction scrap": "Amarillo", "metal/other metal objects": "Amarillo",
    "metal/spray cans": "Rojo",
    "plastic/plastic bags": "Blanco", "plastic/plastic bottles": "Blanco",
    "plastic/plastic containers": "Blanco", "plastic/plastic cups": "Blanco",
    "plastic/cigarette butt": "Negro",
    "glass": "Gris",
    "medical/gloves": "Rojo", "medical/masks": "Rojo",
    "medical/medicines": "Rojo", "medical/syringe": "Rojo",
    "e-waste/electrical cables": "Rojo", "e-waste/electronic chips": "Rojo",
    "e-waste/smartphones": "Rojo", "e-waste/laptops": "Rojo",
    "e-waste/small appliances": "Rojo",
}

## 5. Reunir candidatos (sin duplicados)

In [ ]:
import hashlib
from collections import defaultdict, Counter

def es_imagen(f):
    return os.path.splitext(f)[1].lower() in EXT

def huella(ruta):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()

# candidatos[color][fuente] = [rutas]
candidatos = defaultdict(lambda: defaultdict(list))
vistos = set()
duplicados = 0

def agregar(color, fuente, ruta):
    global duplicados
    hh = huella(ruta)
    if hh in vistos:
        duplicados += 1
        return
    vistos.add(hh)
    candidatos[color][fuente].append(ruta)

# Garbage V2: busca carpetas por nombre en cualquier nivel
for raiz, dirs, archivos in os.walk(RUTA_GD):
    nombre = os.path.basename(raiz).strip().lower()
    if nombre in MAPEO_GD:
        for f in archivos:
            if es_imagen(f):
                agregar(MAPEO_GD[nombre], f"gd/{nombre}", os.path.join(raiz, f))

# TrashBox: por ruta relativa
for raiz, dirs, archivos in os.walk(RUTA_TB):
    rel = os.path.relpath(raiz, RUTA_TB).replace(os.sep, "/").strip().lower()
    if rel in MAPEO_TB:
        for f in archivos:
            if es_imagen(f):
                agregar(MAPEO_TB[rel], f"tb/{rel}", os.path.join(raiz, f))

print(f"Duplicados descartados: {duplicados}\n")
for c in COLORES:
    total = sum(len(v) for v in candidatos[c].values())
    print(f"{c:<9} {total:>6} imágenes")
    for fuente, lista in sorted(candidatos[c].items()):
        print(f"            {fuente:<32} {len(lista):>5}")

## 6. Balancear, separar en splits y guardar
Si un color supera `MAX_POR_COLOR`, se toman imágenes **turnándose entre sus fuentes**
(así Negro no termina siendo solo ropa). Tus fotos propias de Drive se agregan completas, sin tope,
y también se reparten 70/15/15.

In [ ]:
import random, shutil
from PIL import Image, ImageOps

random.seed(SEMILLA)

def elegir_balanceado(por_fuente, tope):
    grupos = [random.sample(l, len(l)) for l in por_fuente.values()]
    elegidas = []
    while len(elegidas) < tope and any(grupos):
        for g in grupos:
            if g and len(elegidas) < tope:
                elegidas.append(g.pop())
    return elegidas

def separar(lista):
    lista = random.sample(lista, len(lista))
    n = len(lista)
    n_train = round(n * PROPORCION["train"])
    n_val = round(n * PROPORCION["val"])
    return {"train": lista[:n_train],
            "val": lista[n_train:n_train + n_val],
            "test": lista[n_train + n_val:]}

def guardar(origen, destino):
    try:
        with Image.open(origen) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((LADO_MAX, LADO_MAX))
            im.save(destino, "JPEG", quality=90)
        return True
    except Exception as e:
        print("  omitida (no se pudo abrir):", origen, e)
        return False

shutil.rmtree(SALIDA_LOCAL, ignore_errors=True)
resumen = defaultdict(Counter)

for c in COLORES:
    descargadas = elegir_balanceado(candidatos[c], MAX_POR_COLOR)
    carpeta_propia = os.path.join(DRIVE_DIR, c)
    propias = []
    if os.path.isdir(carpeta_propia):
        propias = [os.path.join(carpeta_propia, f) for f in os.listdir(carpeta_propia) if es_imagen(f)]

    for origen_tipo, lista in (("dl", descargadas), ("propia", propias)):
        for split, rutas in separar(lista).items():
            dest = os.path.join(SALIDA_LOCAL, split, c)
            os.makedirs(dest, exist_ok=True)
            for i, ruta in enumerate(rutas):
                nombre = f"{origen_tipo}_{c.lower()}_{i:05d}.jpg"
                if guardar(ruta, os.path.join(dest, nombre)):
                    resumen[split][c] += 1
                    if origen_tipo == "propia":
                        resumen[split][c + " (propias)"] += 1
    print(f"{c:<9} descargadas: {len(descargadas):>5} | propias: {len(propias):>4}")

## 7. Resumen de los splits

In [ ]:
import pandas as pd
tabla = pd.DataFrame({s: {c: resumen[s][c] for c in COLORES} for s in ["train", "val", "test"]})
tabla["total"] = tabla.sum(axis=1)
tabla.loc["TOTAL"] = tabla.sum()
print(tabla)
print("\nDe ellas, fotos propias por split:")
print({s: sum(resumen[s][c + ' (propias)'] for c in COLORES) for s in ["train", "val", "test"]})

In [ ]:
# Vista rápida: 4 imágenes de train por color
import matplotlib.pyplot as plt
fig, ejes = plt.subplots(len(COLORES), 4, figsize=(10, 2.6 * len(COLORES)))
for fila, c in enumerate(COLORES):
    carpeta = os.path.join(SALIDA_LOCAL, "train", c)
    muestras = random.sample(os.listdir(carpeta), min(4, len(os.listdir(carpeta))))
    for col in range(4):
        ax = ejes[fila][col]; ax.axis("off")
        if col < len(muestras):
            ax.imshow(Image.open(os.path.join(carpeta, muestras[col])))
        if col == 0:
            ax.set_title(c, loc="left", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

## 8. Guardar en tu Drive
Copia `splits/train`, `splits/val` y `splits/test` dentro de `data basurita`, y además un `.zip` con todo
(el zip es lo más rápido para volver a cargarlo en Colab al entrenar).

In [ ]:
destino_drive = os.path.join(DRIVE_DIR, "splits")
shutil.rmtree(destino_drive, ignore_errors=True)
shutil.copytree(SALIDA_LOCAL, destino_drive)
zip_ruta = shutil.make_archive("/content/dataset_ntp900058", "zip", SALIDA_LOCAL)
shutil.copy(zip_ruta, os.path.join(DRIVE_DIR, "dataset_ntp900058.zip"))
print("Listo:", destino_drive)
print("Zip:", os.path.join(DRIVE_DIR, "dataset_ntp900058.zip"))

## 9. Cómo se carga en PyTorch
La estructura `splits/<split>/<Color>/` es la que lee `torchvision.datasets.ImageFolder` sin cambios:
```python
from torchvision import datasets, transforms
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
train = datasets.ImageFolder("/content/splits/train", transform=tf)
print(train.classes)  # ['Amarillo', 'Azul', 'Blanco', 'Gris', 'Marron', 'Negro', 'Rojo']
```